# 10 — Contextual validation (unblind)

After annotations and axes are frozen, **unblind** sampling cells and test whether
the same topic performs the same function in high- vs low-rated books.

Method: join Pass B `sentence_codes` to evidence-packet `sid→cell`, aggregate
per-cell code proportions, then compare high-prevalence/high-tier vs
high-prevalence/low-tier. No re-prompting; Pass B never saw ratings.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
root = cwd
for _ in range(6):
    if (root / "configs").is_dir() and (root / "src").is_dir():
        break
    root = root.parent
sys.path.insert(0, str(root))

from src.stage11_refined_construct_analysis.analysis import notebook_helpers as nh

ctx = nh.setup("10_contextual_validation")
cfg = ctx.cfg

Project root : /home/polina/Documents/Cursor_Projects/romantic_novels_large_corpus
Config       : configs/stage11/refined_constructs.yaml
Run          : v4_l12_granular_final_call49
Outputs      : results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation


## Unblind cell key

In [2]:
cell_key = nh.load_cell_key(cfg)
print(json.dumps(cell_key, indent=2)[:2000])
ctx.save_markdown(json.dumps(cell_key, indent=2), "cell_key_unblinded")

meanings = cell_key.get("meanings") or cfg.section("evidence", "cell_meanings")
cell_tbl = pd.DataFrame([{"cell": k, "meaning": v} for k, v in meanings.items()])
display(cell_tbl)
ctx.save_table(cell_tbl, "cell_meanings")

{
  "labels": [
    "CELL_A",
    "CELL_B",
    "CELL_C",
    "CELL_D"
  ],
  "meanings": {
    "CELL_A": "high_prevalence_high_tier",
    "CELL_B": "high_prevalence_low_tier",
    "CELL_C": "low_prevalence_high_tier",
    "CELL_D": "low_prevalence_low_tier"
  },
  "sealed": true,
  "note": "Do not open until notebook 10 contextual validation. Pass A/B must only see CELL_* labels; position/tertile may be visible."
}
  saved markdown: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/cell_key_unblinded.md


,cell,meaning
0,CELL_A,high_prevalence_high_tier
1,CELL_B,high_prevalence_low_tier
2,CELL_C,low_prevalence_high_tier
3,CELL_D,low_prevalence_low_tier


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/cell_meanings.csv  (4 rows)


## Cell-level code stability from stored sentence codes

In [3]:
stability = nh.cell_code_stability(cfg)
display(stability.head(20))
ctx.save_table(stability, "pass_b_cell_stability_flags")

if not stability.empty:
    by_hyp = (
        stability.groupby("hypothesis")
        .agg(
            n_topics=("topic_id", "nunique"),
            n_with_both_high_prev=(
                "meaning_differs_high_prevalence",
                lambda s: int(s.notna().sum()),
            ),
            n_differs=(
                "meaning_differs_high_prevalence",
                lambda s: int((s == True).sum()),  # noqa: E712
            ),
            pct_differs=(
                "meaning_differs_high_prevalence",
                lambda s: float((s == True).mean()) if s.notna().any() else float("nan"),  # noqa: E712
            ),
        )
        .reset_index()
    )
    display(by_hyp.round(4))
    ctx.save_table(by_hyp, "cell_stability_by_hypothesis")

    drifted = stability[stability["meaning_differs_high_prevalence"] == True]  # noqa: E712
    if not drifted.empty:
        show = drifted[
            [
                "hypothesis",
                "topic_id",
                "high_prev_high_tier_code",
                "high_prev_low_tier_code",
                "pass_b_dominant",
            ]
        ]
        display(show)
        ctx.save_table(show, "topics_with_high_prev_code_drift")
    print(
        f"Topics with comparable high-prevalence cells: "
        f"{int(stability['meaning_differs_high_prevalence'].notna().sum())}; "
        f"dominant-code drift: "
        f"{int((stability['meaning_differs_high_prevalence'] == True).sum())}"  # noqa: E712
    )
else:
    print("No sentence_codes × cell joins available.")

,hypothesis,topic_id,n_coded_sentences,n_cells_with_codes,dominant_by_cell,dominant_by_meaning,high_prev_high_tier_code,high_prev_low_tier_code,meaning_differs_high_prevalence,pass_b_dominant
0,H1,1,20,3,"{""CELL_D"": ""I9"", ""CELL_A"": ""I3"", ""CELL_C"": ""I6""}","{""low_prevalence_low_tier"": ""I9"", ""high_preval...",I3,None,None,MIXED
1,H1,7,20,2,"{""CELL_D"": ""I3"", ""CELL_A"": ""I3""}","{""low_prevalence_low_tier"": ""I3"", ""high_preval...",I3,None,None,I3
2,H1,11,20,3,"{""CELL_A"": ""I10"", ""CELL_B"": ""I10"", ""CELL_D"": ""...","{""high_prevalence_high_tier"": ""I10"", ""high_pre...",I10,I10,False,I10
3,H1,19,20,3,"{""CELL_B"": ""I3"", ""CELL_C"": ""I3"", ""CELL_A"": ""I3""}","{""high_prevalence_low_tier"": ""I3"", ""low_preval...",I3,I3,False,I3
4,H1,24,20,3,"{""CELL_D"": ""I0"", ""CELL_B"": ""I0"", ""CELL_A"": ""I0""}","{""low_prevalence_low_tier"": ""I0"", ""high_preval...",I0,I0,False,I0
5,H1,25,20,3,"{""CELL_D"": ""I0"", ""CELL_A"": ""I0"", ""CELL_B"": ""I3""}","{""low_prevalence_low_tier"": ""I0"", ""high_preval...",I0,I3,True,I0
6,H1,29,20,3,"{""CELL_B"": ""I1"", ""CELL_D"": ""I1"", ""CELL_C"": ""I1""}","{""high_prevalence_low_tier"": ""I1"", ""low_preval...",None,I1,None,I1
7,H1,31,20,3,"{""CELL_D"": ""I0"", ""CELL_B"": ""I0"", ""CELL_C"": ""I0""}","{""low_prevalence_low_tier"": ""I0"", ""high_preval...",None,I0,None,I0
8,H1,36,20,3,"{""CELL_B"": ""I2"", ""CELL_D"": ""I2"", ""CELL_C"": ""I2""}","{""high_prevalence_low_tier"": ""I2"", ""low_preval...",None,I2,None,I2
9,H1,37,20,3,"{""CELL_B"": ""I1"", ""CELL_D"": ""I1"", ""CELL_C"": ""I1""}","{""high_prevalence_low_tier"": ""I1"", ""low_preval...",None,I1,None,I1


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/pass_b_cell_stability_flags.csv  (272 rows)


,hypothesis,n_topics,n_with_both_high_prev,n_differs,pct_differs
0,H1,97,49,7,0.0722
1,H2,10,4,0,0.0000
2,H3,82,41,11,0.1341
3,H4,32,16,2,0.0625
4,H5,22,13,0,0.0000
5,H6,29,19,3,0.1034


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/cell_stability_by_hypothesis.csv  (6 rows)


,hypothesis,topic_id,high_prev_high_tier_code,high_prev_low_tier_code,pass_b_dominant
5,H1,25,I0,I3,I0
11,H1,41,I6,I3,MIXED
35,H1,101,I0,I3,MIXED
48,H1,156,I8,I6,MIXED
79,H1,299,I0,I2,I0
85,H1,340,I0,I4,MIXED
93,H1,365,I0,I2,MIXED
116,H3,46,S0,S3,S3
136,H3,116,S0,S5,S0
147,H3,172,S7,S15,S7


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/topics_with_high_prev_code_drift.csv  (23 rows)
Topics with comparable high-prevalence cells: 142; dominant-code drift: 23


## High vs low construct × high vs low rating (book-level)

In [4]:
frame = nh.load_refined_frame(cfg, "strict")
usable = frame[frame["analysable"].fillna(True)] if "analysable" in frame.columns else frame
constructs = [
    c
    for c in (
        "RAX_nonexplicit_affection",
        "RAX_explicit_sex",
        "RAX_h2_strict",
        "RAX_emotional_security",
        "RAX_appearance_grooming",
        "RAX_status_display",
        "RAX_external_protection",
        "RAX_external_danger_crisis",
        "RAX_relational_darkness",
        "RARC",
    )
    if c in usable.columns
]

cell_rows = []
for c in constructs:
    q_lo, q_hi = usable[c].quantile(0.25), usable[c].quantile(0.75)
    for tier in ("high_rate", "low_rate"):
        for level, mask in (
            ("high_construct", usable[c] >= q_hi),
            ("low_construct", usable[c] <= q_lo),
        ):
            sub = usable.loc[mask & (usable["rating_class"] == tier), c]
            cell_rows.append(
                {
                    "construct": c,
                    "cell": f"{level}×{tier}",
                    "n_books": int(len(sub)),
                    "mean_share": float(sub.mean()) if len(sub) else float("nan"),
                }
            )
cells = pd.DataFrame(cell_rows)
display(cells)
ctx.save_table(cells, "construct_x_rating_cells")

print(
    "Interpretive questions for close reading (use human_review packets):\n"
    "- When a topic is strongly present, does its function match in high- vs low-rated books?\n"
    "- Is appearance/grooming distinct from status display in both rating tiers?\n"
    "- Does 'protection' in low-rated books look more like control?"
)

,construct,cell,n_books,mean_share
0,RAX_nonexplicit_affection,high_construct×high_rate,1319,0.0272
1,RAX_nonexplicit_affection,low_construct×high_rate,1204,0.0030
2,RAX_nonexplicit_affection,high_construct×low_rate,1315,0.0276
3,RAX_nonexplicit_affection,low_construct×low_rate,1410,0.0028
4,RAX_explicit_sex,high_construct×high_rate,1084,0.0156
5,RAX_explicit_sex,low_construct×high_rate,1354,0.0013
6,RAX_explicit_sex,high_construct×low_rate,1583,0.0193
7,RAX_explicit_sex,low_construct×low_rate,1324,0.0014
8,RAX_h2_strict,high_construct×high_rate,5086,0.0000
9,RAX_h2_strict,low_construct×high_rate,5086,0.0000


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/10_contextual_validation/tables/construct_x_rating_cells.csv  (40 rows)
Interpretive questions for close reading (use human_review packets):
- When a topic is strongly present, does its function match in high- vs low-rated books?
- Is appearance/grooming distinct from status display in both rating tiers?
- Does 'protection' in low-rated books look more like control?
